# CivicLens AI — RDD2022 India YOLO11 training
Run this notebook in **Google Colab with a GPU runtime**. It downloads the published RDD2022 archive, prepares the labelled India images, fine-tunes YOLO11n, evaluates a held-out split, and exports ONNX. It does not claim NMC integration or Nagpur field validation.

**Before starting:** Runtime → Change runtime type → T4 GPU (or better). Training may take several hours depending on GPU and epochs.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/ayush0531/civiclens-ai.git
%cd civiclens-ai
!git checkout main
!pip install -q -r ml/requirements.txt

## Download the published RDD2022 archive
The archive contains labelled training images and unlabelled challenge test images. We use only labelled India images for the train/validation/held-out split; we do not report metrics on unlabelled challenge images. The archive is large; ensure your Colab session has enough disk space.

In [ ]:
from pathlib import Path
import urllib.request, zipfile, os

archive = Path('/content/RDD2022.zip')
url = 'https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/RDD2022.zip'
if not archive.exists():
    print('Downloading RDD2022 archive (large download; please wait)...')
    urllib.request.urlretrieve(url, archive)
print(f'Archive size: {archive.stat().st_size / (1024**3):.2f} GiB')
with zipfile.ZipFile(archive) as z:
    names = z.namelist()
    india = [n for n in names if 'India' in n and (n.lower().endswith(('.jpg','.jpeg','.png','.xml')))]
    print('India image/annotation files in archive:', len(india))
    if not india: raise RuntimeError('Could not find India files in archive. Inspect z.namelist() and update extraction filter.')
    z.extractall('/content/RDD2022', members=india)
print('India subset extracted.')

In [ ]:
from pathlib import Path
root = Path('/content/RDD2022')
xmls = list(root.rglob('*.xml'))
print('India XML annotations:', len(xmls))
print('Example folders:', [str(p) for p in list(root.iterdir())[:10]])
if len(xmls) < 100: raise RuntimeError('Too few India XML annotations were extracted; inspect archive directory names.')
!python ml/prepare_rdd2022.py --source /content/RDD2022 --output /content/datasets/rdd2022-india

## Train YOLO11n
Default: 100 epochs, 640px images. For a short smoke test, set `EPOCHS = 2`; do not report smoke-test metrics as final model performance.

In [ ]:
EPOCHS = 100
BATCH = 16
!python ml/train_yolo11.py --data /content/datasets/rdd2022-india/data.yaml --epochs {EPOCHS} --imgsz 640 --batch {BATCH} --device 0 --workers 8

## Review outputs and export artifacts
Inspect the confusion matrix, PR curves, per-class AP and sample predictions before claiming validation. Download `best.pt`, `best.onnx`, and `evaluation_summary.json` from the run folder. The held-out split is an internal RDD2022 India split, not independent Nagpur field validation.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
runs = Path('/content/civiclens-ai/ml/runs')
summaries = list(runs.rglob('evaluation_summary.json'))
print('Evaluation summaries:', summaries)
if summaries:
    print(summaries[-1].read_text())
for p in runs.rglob('confusion_matrix.png'):
    display(Image(filename=str(p)))
for p in runs.rglob('PR_curve.png'):
    display(Image(filename=str(p)))

In [ ]:
from google.colab import files
artifact_dir = summaries[-1].parent if summaries else None
if artifact_dir:
    for name in ['weights/best.pt', 'weights/best.onnx', 'evaluation_summary.json']:
        p = artifact_dir / name
        if p.exists(): files.download(str(p))
        else: print('Not found:', p)